In [ ]:
from datasets import load_dataset
ds = load_dataset("Data-Science-Nigeria/dsn-2026-lid-train", streaming=True)
print(next(iter(ds["train"])))

In [ ]:
!pip install -q transformers accelerate datasets[audio] soundfile librosa audiomentations scikit-learn huggingface_hub safetensors

import os, io, json, time, zipfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import librosa
import soundfile as sf
from datasets import load_dataset, Audio
from huggingface_hub import login
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import (
    AutoConfig,
    AutoFeatureExtractor,
    AutoModelForAudioClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)
from audiomentations import Compose, AddGaussianNoise, Gain, PitchShift, LowPassFilter

warnings.filterwarnings("ignore")
set_seed(42)

# Paste your Hugging Face Write Token here (or load from Kaggle Secrets)
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

OFFICIAL_LABELS = [
    "yor", "ibo", "hau", "ego", "anw", "efi",
    "ibb", "idu", "igl", "nup", "tiv", "urh", "other"
]
LABEL2ID = {label: i for i, label in enumerate(OFFICIAL_LABELS)}
ID2LABEL = {i: label for i, label in enumerate(OFFICIAL_LABELS)}
TARGET_SR = 16000
MAX_DURATION_SEC = 10.0

# Load from local /kaggle/working/dsn_2026_lid_train if present, otherwise directly from Hugging Face Hub
LOCAL_DIR = Path("/kaggle/working/dsn_2026_lid_train")
if (LOCAL_DIR / "data").exists():
    print("Loading Parquet dataset from /kaggle/working/dsn_2026_lid_train...")
    ds = load_dataset("parquet", data_files={
        "train": str(LOCAL_DIR / "data/train-*.parquet"),
        "validation": str(LOCAL_DIR / "data/validation-*.parquet")
    })
else:
    print("Downloading Data-Science-Nigeria/dsn-2026-lid-train from Hugging Face Hub...")
    ds = load_dataset("Data-Science-Nigeria/dsn-2026-lid-train", token=HF_TOKEN)

# Cast audio column so we decode cleanly at 16 kHz
ds = ds.cast_column("audio", Audio(sampling_rate=TARGET_SR, decode=True))
print(ds)

In [ ]:
ds = load_dataset("Data-Science-Nigeria/dsn-2026-lid-train", token=HF_TOKEN)
ds = ds.cast_column("audio", Audio(sampling_rate=TARGET_SR, decode=True))
print(ds)

In [ ]:
from datasets import concatenate_datasets

train_hf = ds["train"]
val_hf = ds["validation"]

# Find indices of Nupe ('nup') in train (176 clips total)
labels_list = train_hf["label"]
nup_indices = [i for i, lbl in enumerate(labels_list) if lbl == "nup"]
other_indices = [i for i, lbl in enumerate(labels_list) if lbl != "nup"]

# Move 26 Nupe clips to Validation so all 13 classes exist in Validation
np.random.seed(42)
np.random.shuffle(nup_indices)
nup_val_idx = nup_indices[:26]
nup_train_idx = nup_indices[26:]

nup_val_subset = train_hf.select(nup_val_idx)
train_hf = train_hf.select(other_indices + nup_train_idx)
val_hf = concatenate_datasets([val_hf, nup_val_subset])

print(f"Adjusted Train size: {len(train_hf):,} | Adjusted Val size (with Nupe): {len(val_hf):,}")

# Augmenter: includes PitchShift specifically so 1-2 speaker classes (nup, igl, ibb) don't memorize speaker pitch
train_augmenter = Compose([
    PitchShift(min_semitones=-2.5, max_semitones=2.5, p=0.5),
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.4),
    Gain(min_gain_db=-6.0, max_gain_db=6.0, p=0.4),
    LowPassFilter(min_cutoff_freq=3000, max_cutoff_freq=4000, p=0.3), # Simulates phone-quality test audio
])

val_nup_augmenter = PitchShift(min_semitones=2.0, max_semitones=2.5, p=1.0)

class HFParquetLIDDataset(Dataset):
    def __init__(self, hf_split, feature_extractor, max_duration=MAX_DURATION_SEC, is_train=True):
        self.split = hf_split
        self.fe = feature_extractor
        self.max_len = int(max_duration * TARGET_SR)
        self.is_train = is_train

    def __len__(self):
        return len(self.split)

    def __getitem__(self, idx):
        item_raw = self.split[idx]
        audio = np.asarray(item_raw["audio"]["array"], dtype=np.float32)
        label_str = item_raw["label"]
        target_id = LABEL2ID[label_str]

        if len(audio) > self.max_len:
            start = np.random.randint(0, len(audio) - self.max_len) if self.is_train else (len(audio) - self.max_len) // 2
            audio = audio[start : start + self.max_len]

        if self.is_train:
            audio = train_augmenter(samples=audio, sample_rate=TARGET_SR)
        elif label_str == "nup":
            # Perturb pitch on held-out Nupe clips so validation tests language, not speaker identity
            audio = val_nup_augmenter(samples=audio, sample_rate=TARGET_SR)

        inputs = self.fe(
            audio,
            sampling_rate=TARGET_SR,
            max_length=self.max_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )
        out = {k: v.squeeze(0) for k, v in inputs.items()}
        out["labels"] = torch.tensor(target_id, dtype=torch.long)
        return out

# Compute dampened class weights (power 0.5 prevents extreme instability on 170-clip classes)
train_target_ids = np.array([LABEL2ID[l] for l in train_hf["label"]])
class_counts = np.bincount(train_target_ids, minlength=len(OFFICIAL_LABELS))
raw_weights = len(train_target_ids) / (len(OFFICIAL_LABELS) * np.maximum(class_counts, 1))
dampened_weights = np.power(raw_weights, 0.5)
dampened_weights = dampened_weights / dampened_weights.mean()
CLASS_WEIGHTS_TENSOR = torch.tensor(dampened_weights, dtype=torch.float32).cuda()

# Sample weights for WeightedRandomSampler (oversamples igl, nup, efi, ibb every epoch)
sample_weights = torch.tensor([dampened_weights[t] for t in train_target_ids], dtype=torch.double)

class OversampledWeightedTrainer(Trainer):
    def _get_train_sampler(self, train_dataset=None):
        return WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS_TENSOR.to(logits.device), label_smoothing=0.05)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [ ]:
# ============================================================
# DSN 2026 LANGUAGE IDENTIFICATION
# MODEL: WavLM Base Plus (ONLY)
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from sklearn.metrics import f1_score
from transformers import (
    AutoFeatureExtractor,
    AutoConfig,
    AutoModelForAudioClassification,
    TrainingArguments,
    EarlyStoppingCallback,
)

# ------------------------------------------------------------
# 1. Check GPU
# ------------------------------------------------------------

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Enable a GPU accelerator before training."
    )

print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# 2. Official label configuration
# ------------------------------------------------------------
# OFFICIAL_LABELS, LABEL2ID, and ID2LABEL must already be
# defined using the official DSN competition label mapping.

assert len(OFFICIAL_LABELS) == len(LABEL2ID)
assert len(OFFICIAL_LABELS) == len(ID2LABEL)

# ------------------------------------------------------------
# 3. Evaluation metrics
# ------------------------------------------------------------

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "macro_f1": f1_score(
            labels, preds, average="macro", zero_division=0
        ),
        "weighted_f1": f1_score(
            labels, preds, average="weighted", zero_division=0
        ),
    }

# ------------------------------------------------------------
# 4. Calibrate classifier bias using validation predictions
# ------------------------------------------------------------

def calibrate_classifier_bias(model, val_logits, val_labels):
    num_classes = len(OFFICIAL_LABELS)
    best_offsets = np.zeros(num_classes, dtype=np.float32)

    base_f1 = f1_score(
        val_labels,
        np.argmax(val_logits, axis=-1),
        average="macro",
        zero_division=0,
    )
    best_f1 = base_f1

    for _ in range(3):
        for c in range(num_classes):
            for delta in np.linspace(-1.5, 1.5, 31):
                trial_offsets = best_offsets.copy()
                trial_offsets[c] = delta

                preds = np.argmax(
                    val_logits + trial_offsets, axis=-1
                )
                score = f1_score(
                    val_labels,
                    preds,
                    average="macro",
                    zero_division=0,
                )

                if score > best_f1:
                    best_f1 = score
                    best_offsets = trial_offsets

    print(
        f"Validation Macro F1: {base_f1:.4f} -> {best_f1:.4f}"
    )

    # WavLM's classification head uses classifier.out_proj.
    # Apply offsets to the actual final classification layer.
    classifier = getattr(model, "classifier", None)
    output_layer = getattr(classifier, "out_proj", None)

    if output_layer is None and getattr(classifier, "bias", None) is not None:
        output_layer = classifier

    if output_layer is not None and output_layer.bias is not None:
        with torch.no_grad():
            offsets = torch.tensor(
                best_offsets,
                device=output_layer.bias.device,
                dtype=output_layer.bias.dtype,
            )
            output_layer.bias.add_(offsets)
    else:
        print(
            "Warning: Could not locate the classifier bias. "
            "Offsets were calculated but not applied to the model."
        )

    return best_f1, best_offsets

# ------------------------------------------------------------
# 5. Load ONLY WavLM Base Plus
# ------------------------------------------------------------

MODEL_NAME = "wavlm-base-plus"
MODEL_ID = "microsoft/wavlm-base-plus"

OUTPUT_ROOT = Path("/content/drive/MyDrive/lid_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_DIR = OUTPUT_ROOT / MODEL_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
EXPORT_DIR = RUN_DIR / "submission_repo"

print(f"\nLoading pretrained model: {MODEL_ID}")

feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)

config = AutoConfig.from_pretrained(
    MODEL_ID,
    num_labels=len(OFFICIAL_LABELS),
    label2id=LABEL2ID,
    id2label=ID2LABEL,
)

model = AutoModelForAudioClassification.from_pretrained(
    MODEL_ID,
    config=config,
    ignore_mismatched_sizes=True,
)

if hasattr(model, "freeze_feature_encoder"):
    model.freeze_feature_encoder()

print("WavLM Base Plus loaded successfully.")

# ------------------------------------------------------------
# 6. Prepare training and validation datasets
# ------------------------------------------------------------
# These objects must have been created in your data-preparation
# cells. Your dataset must return input_values and labels.

train_ds = HFParquetLIDDataset(
    train_hf, feature_extractor, is_train=True
)

val_ds = HFParquetLIDDataset(
    val_hf, feature_extractor, is_train=False
)

print("Training examples:", len(train_ds))
print("Validation examples:", len(val_ds))

# ------------------------------------------------------------
# 7. Training configuration
# ------------------------------------------------------------

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=4,
    warmup_steps=100,
    weight_decay=0.01,
    fp16=True,
    bf16=False,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none",
    seed=42,
    dataloader_num_workers=2,
)

# ------------------------------------------------------------
# 8. Create trainer
# ------------------------------------------------------------
# OversampledWeightedTrainer must be defined in an earlier cell.

trainer = OversampledWeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=feature_extractor,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2)
    ],
)

# ------------------------------------------------------------
# 9. Train the ONE model
# ------------------------------------------------------------

print("\nStarting WavLM Base Plus training...")

trainer.train()

print("\nTraining completed.")

# ------------------------------------------------------------
# 10. Evaluate on validation data
# ------------------------------------------------------------

print("\nEvaluating on validation data...")

pred_out = trainer.predict(val_ds)

val_logits = pred_out.predictions
val_labels = pred_out.label_ids

raw_macro_f1 = f1_score(
    val_labels,
    np.argmax(val_logits, axis=-1),
    average="macro",
    zero_division=0,
)

print(f"Raw validation Macro F1: {raw_macro_f1:.4f}")
print("Validation metrics:", pred_out.metrics)

# ------------------------------------------------------------
# 11. Calibrate classifier bias
# ------------------------------------------------------------

calibrated_f1, offsets = calibrate_classifier_bias(
    trainer.model,
    val_logits,
    val_labels,
)

# ------------------------------------------------------------
# 12. Save model in FP16 format
# ------------------------------------------------------------

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

trainer.model.to(torch.float16)

trainer.model.save_pretrained(
    EXPORT_DIR,
    safe_serialization=True,
)

feature_extractor.save_pretrained(EXPORT_DIR)

# Save calibration offsets for reference and inference.
np.save(
    EXPORT_DIR / "calibration_offsets.npy",
    offsets,
)

# ------------------------------------------------------------
# 13. Print final results
# ------------------------------------------------------------

num_params = sum(
    p.numel() for p in trainer.model.parameters()
) / 1e6

repo_gb = sum(
    f.stat().st_size
    for f in EXPORT_DIR.rglob("*")
    if f.is_file()
) / (1024 ** 3)

print("\n========== FINAL RESULTS ==========")
print("Model: WavLM Base Plus")
print(f"Raw Macro F1: {raw_macro_f1:.4f}")
print(f"Calibrated validation Macro F1: {calibrated_f1:.4f}")
print(f"Parameters: {num_params:.2f} million")
print(f"Export size: {repo_gb:.3f} GB")
print(f"Saved model: {EXPORT_DIR}")
print("===================================")


In [ ]:
# ============================================================
# UPLOAD MODEL TO HUGGING FACE + CREATE SUBMISSION ZIP
# GOOGLE COLAB VERSION
# ============================================================

!pip -q install -U huggingface_hub

from huggingface_hub import HfApi
from google.colab import userdata, files
import json
import zipfile
import os

# 1. Hugging Face configuration
try:
    HF_TOKEN = globals().get("HF_TOKEN") or userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

REPO_ID = "AbdulroqeebA/wavlm-base-plus-nigerian-lid"

# Use the folder containing your trained model.
export_dir = "/content/drive/MyDrive/lid_outputs/wavlm-base-plus/submission_repo"

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is missing. Please make sure to login to Hugging Face or save your token in Colab Secrets.")

if not os.path.isdir(export_dir):
    raise FileNotFoundError(
        f"Model folder not found: {export_dir}\n"
        "Update export_dir to your actual model folder."
    )

# 2. Create or reuse the Hugging Face model repository
api = HfApi(token=HF_TOKEN)

api.create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    private=False,
    exist_ok=True,
)

# 3. Upload the trained model
commit_info = api.upload_folder(
    folder_path=export_dir,
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="Upload fine-tuned audio language identification model",
)

full_commit_hash = commit_info.oid

print(f"Model uploaded: https://huggingface.co/{REPO_ID}")
print(f"Full commit hash: {full_commit_hash}")

# 4. Create submission.json
sub_payload = {
    "repo": REPO_ID,
    "revision": full_commit_hash,
}

submission_json = "/content/submission.json"

with open(submission_json, "w", encoding="utf-8") as f:
    json.dump(sub_payload, f, indent=2)

# 5. Create submission.zip
submission_zip = "/content/submission.zip"

with zipfile.ZipFile(
    submission_zip, "w", compression=zipfile.ZIP_DEFLATED
) as zf:
    zf.write(submission_json, arcname="submission.json")

print("\nSubmission files created successfully:")
print(submission_json)
print(submission_zip)

# 6. Download the ZIP to your computer
files.download(submission_zip)